# TradeSense · TGNN branch · Tardis.dev download and audit (BTCUSDT, Binance Spot, 2024-05-01) → `TGNN_Tardis_Binance_Spot_BTCUSDT_2024-05-01_v1`

**Read this first (2 minutes).**

**What this notebook does** (and nothing more): downloads **three files for one symbol and one day** (BTCUSDT, 2024-05-01) from Tardis.dev's *free first-day-of-month* area, then audits them: file integrity, schemas, timestamps, coverage, snapshot boundaries, whether an order book can be rebuilt, how often the rebuilt book is crossed, and whether trades and book timestamps agree.

**What it does NOT do:** no other dates, no ETHUSDT, no synthetic manipulation events, no graphs, no training, and it never writes inside any existing research folder (Layer 1, Layer 3, models, the Hugging Face dataset). Existing files are fingerprinted before and after to prove this.

**Official download method (read from docs.tardis.dev, 2026-10-12):** a plain HTTPS GET of
`https://datasets.tardis.dev/v1/{exchange}/{data_type}/{YYYY}/{MM}/{DD}/{SYMBOL}.csv.gz` with **no API key** for the first day of each month. Files are gzipped CSV; `timestamp` and `local_timestamp` are in **microseconds** since epoch (UTC).

**Licence summary (Tardis Terms of Service, read via a web tool, NOT legal advice — please read the original: https://docs.tardis.dev/legal/terms-of-service):**
* "Permitted Use" is *internal business, research, educational or personal use* — so a final-year research project is within the stated scope. Free/sample data downloaded under free access is covered by the same licence (clause 9.4).
* Clause 9.5 allows you to *develop, train and use task-specific quantitative or statistical models for your own market research* (internal use). It excludes general-purpose / foundation / large-language models. A small TGNN for one task is task-specific, but **you must not give trained models to third parties**.
* **You must not redistribute the raw data** (clause 9.2): do not upload these files to GitHub, Kaggle, a public Drive link, etc. Keep the Drive folder private. This notebook's share block contains only aggregate statistics, no raw rows.
* The terms do **not** say whether publishing results in a thesis/paper is allowed. Ask Tardis in writing before publishing, and ask your supervisor how examiners may access results without receiving raw data or model files.

**How the safety switch works:** the download cell refuses to run until you set `I_HAVE_READ_THE_TERMS = True` (cell 2). Until then the notebook only checks free disk space and asks the server how big the files are (no data is downloaded).

**How to run:** Runtime → Run all. If it stops at the gate, read the message, change the one flag, and Run all again.

**Three kinds of checks — kept strictly separate (a green notebook is NOT a validated book):**
1. **Notebook execution tests (E01…)** — did the code do what it should (nothing existing modified, only the intended files downloaded, replay deterministic, the book inspected only after completed messages)?
2. **Dataset integrity tests (D01…)** — are the downloaded files intact and well-formed (sizes, gzip, schema, row counts, units, sane values)? If any of these fail, the folder is NOT finalised.
3. **Order-book reconstruction assessment (R01…)** — how well does the rebuilt book agree with Tardis' own `book_snapshot_25` at aligned checkpoints, and how often is it crossed/locked? Criteria are frozen below. Passing them means *"consistent with the data provider's own snapshots"*. **It is not independent validation:** `incremental_book_L2` and `book_snapshot_25` are produced by the same provider from the same feed, and no historical exchange-side ground truth exists for 2024.

**Frozen reconstruction criteria (written before any data is seen):** R01 crossed+locked share of completed messages ≤ 0.5 %; R02 top-25 price agreement ≥ 99 % of compared levels; R03 top-25 quantity agreement ≥ 99 %; R04 at least 500 aligned snapshot checkpoints (otherwise the comparison is too thin to say anything); R05 ≥ 99 % of the 1,440 minutes contain a book message. Trade-price behaviour is reported as a **diagnostic only** (it never passes or fails the reconstruction): trades legitimately print at the best bid/ask, and the trade and depth feeds are separate WebSocket connections with different latencies.

## 1 · Setup, folders and a fingerprint of everything that already exists

In [ ]:
import os, sys, io, re, gc, json, time, math, gzip, uuid, shutil, hashlib, heapq, warnings, resource
from array import array
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import numpy as np
import pandas as pd
import requests
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.compute as pc
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 100)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
assert BASE.is_dir(), f'Drive folder not found: {BASE}'
EXCHANGE, SYMBOL, DAY = 'binance', 'BTCUSDT', '2024-05-01'
Y, M, D = DAY.split('-')
TYPES = {'incremental_book_L2': True, 'trades': True, 'book_snapshot_25': False}          # name -> required?
BASE_URL = os.environ.get('TARDIS_BASE_URL', 'https://datasets.tardis.dev')                # env var only used by my mock-server tests
OUT_FINAL = BASE / 'TGNN_Tardis_Binance_Spot_BTCUSDT_2024-05-01_v1'
STAGE = BASE / ('.tardis_stage_' + uuid.uuid4().hex[:8])
assert not OUT_FINAL.exists(), f'REFUSING to overwrite {OUT_FINAL}'
UTC = lambda: datetime.now(timezone.utc).isoformat(); RUN_STARTED = UTC(); T_ALL = time.time(); STAGES = {}; REPORT = {}
def sha256_file(p, chunk=16 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
def dump_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(obj, indent=2, default=jsonable), encoding='utf-8')
def peak_mem_gb(): return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6      # Linux: KB -> GB
def tick(name, t0): STAGES[name] = {'seconds': round(time.time() - t0, 1), 'peak_mem_gb_so_far': round(peak_mem_gb(), 2)}; print(f'[{name}] {STAGES[name]["seconds"]} s | peak memory so far {STAGES[name]["peak_mem_gb_so_far"]} GB')
# fingerprint (size + modified time, no hashing) of EVERYTHING already inside the research-datasets folder
t0 = time.time(); PIN0 = {str(p.relative_to(BASE)): (p.stat().st_size, p.stat().st_mtime_ns) for p in BASE.rglob('*') if p.is_file()}
print(f'fingerprinted {len(PIN0)} existing files (they will be re-checked at the end)'); tick('fingerprint', t0)
HEADERS = {'User-Agent': 'TradeSense-research-notebook/1.0 (final-year project)'}
API_KEY = None            # leave as None: the first day of each month needs no key. Only set one if you bought a subscription.
if API_KEY: HEADERS['Authorization'] = 'Bearer ' + API_KEY
def url_for(dt): return f'{BASE_URL}/v1/{EXCHANGE}/{dt}/{Y}/{M}/{D}/{SYMBOL}.csv.gz'
def fname_for(dt): return f'{EXCHANGE}_{dt}_{DAY}_{SYMBOL}.csv.gz'
for dt in TYPES: print(dt, '->', url_for(dt))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fingerprinted 750 existing files (they will be re-checked at the end)
[fingerprint] 0.9 s | peak memory so far 0.18 GB
incremental_book_L2 -> https://datasets.tardis.dev/v1/binance/incremental_book_L2/2024/05/01/BTCUSDT.csv.gz
trades -> https://datasets.tardis.dev/v1/binance/trades/2024/05/01/BTCUSDT.csv.gz
book_snapshot_25 -> https://datasets.tardis.dev/v1/binance/book_snapshot_25/2024/05/01/BTCUSDT.csv.gz


## 2 · Safety gate, free disk space, and "how big are the files?" (still no data downloaded)

In [ ]:
I_HAVE_READ_THE_TERMS = True      # <-- change to True ONLY after you have read the Tardis Terms of Service (link in the first cell)
INCLUDE_BOOK_SNAPSHOT_25 = True    # optional third file; used to cross-check the rebuilt book
MAX_TOTAL_GB = 8.0                 # safety limit for the three files together

du = shutil.disk_usage(str(BASE)); free_gb = du.free / 1e9
loc = shutil.disk_usage('/content' if Path('/content').exists() else '/tmp')
print(f'Drive/target volume: total {du.total/1e9:.1f} GB | free {free_gb:.1f} GB   ||  local disk free {loc.free/1e9:.1f} GB')
REPORT['storage'] = {'target_total_gb': du.total / 1e9, 'target_free_gb': free_gb, 'local_free_gb': loc.free / 1e9}

def probe(url):
    # Plain GET with stream=True: only the response HEADERS are read, then the connection is closed (no file body is downloaded).
    # (A HEAD request can be answered differently from GET by some servers, so we do not rely on it.)
    out = {'url': url, 'status': None, 'bytes': None, 'content_type': None, 'last_modified': None, 'error': None, 'body': None}
    try:
        r = requests.get(url, headers=HEADERS, stream=True, allow_redirects=True, timeout=60); out['status'] = r.status_code
        out['content_type'] = r.headers.get('Content-Type'); out['last_modified'] = r.headers.get('Last-Modified')
        if r.status_code in (200, 206): out['bytes'] = int(r.headers['Content-Length']) if 'Content-Length' in r.headers else None
        else: out['body'] = r.raw.read(400, decode_content=True).decode('utf-8', 'replace')    # small error message from the server, if any
        r.close()
    except Exception as e: out['error'] = type(e).__name__ + ': ' + str(e)[:200]
    return out
PROBE = {dt: probe(url_for(dt)) for dt in TYPES if TYPES[dt] or INCLUDE_BOOK_SNAPSHOT_25}
display(pd.DataFrame(PROBE).T[['status', 'bytes', 'content_type', 'last_modified', 'error', 'body']])
REPORT['preflight'] = PROBE
known = [v['bytes'] for v in PROBE.values() if v['bytes']]; total_gb = sum(known) / 1e9
bad = {k: v for k, v in PROBE.items() if v['status'] not in (200, 206) or (v['bytes'] is not None and v['bytes'] < 1000)}    # tiny file = Tardis' empty-gzip answer for 'no data'
print(f'\nTotal compressed size of the files above: {total_gb:.2f} GB (files with unknown size: {sum(1 for v in PROBE.values() if not v["bytes"])})')
if bad:
    print('\n!! The server did not return the file for:', {k: (v['status'], v['error'], v['body']) for k, v in bad.items()})
    print('   URLs tried:'); [print('    ', v['url']) for v in bad.values()]
    try:    # read-only metadata call (no dataset download): what does Tardis itself list for this exchange/symbol?
        mj = requests.get(f'https://api.tardis.dev/v1/exchanges/{EXCHANGE}', timeout=45).json(); ds = mj.get('datasets') or {}
        sy = [x for x in ds.get('symbols', []) if str(x.get('id', '')).upper() == SYMBOL]
        print('   Tardis metadata -> supportsDatasets:', mj.get('supportsDatasets'), '| exportedUntil:', ds.get('exportedUntil'), '| symbol entry:', sy[:1] if sy else 'NOT LISTED in datasets.symbols')
        REPORT['tardis_metadata_check'] = {'supportsDatasets': mj.get('supportsDatasets'), 'exportedUntil': ds.get('exportedUntil'), 'symbol_entry': sy[:1]}
    except Exception as e: print('   (metadata check failed:', type(e).__name__, str(e)[:120], ')')
    print('   404 = file does not exist for that date/type; 401/403 = free access does not apply to this file. Do NOT try to work around this; paste this output to Claude.')
    raise RuntimeError('STOPPED ON PURPOSE: availability check failed (see message above). Nothing was downloaded.')
if total_gb > MAX_TOTAL_GB: raise RuntimeError(f'STOPPED ON PURPOSE: files total {total_gb:.1f} GB > MAX_TOTAL_GB={MAX_TOTAL_GB}. Raise the limit only if you have the space.')
if total_gb * 1.2 + 2 > free_gb: raise RuntimeError(f'STOPPED ON PURPOSE: not enough free space ({free_gb:.1f} GB free, need about {total_gb*1.2+2:.1f} GB).')
if not I_HAVE_READ_THE_TERMS:
    print('\n' + '=' * 90 + '\nGATE: availability OK and space OK. NOTHING HAS BEEN DOWNLOADED.\nTo continue: read the Tardis Terms of Service (link in the first cell), set I_HAVE_READ_THE_TERMS = True above, then Run all.\n' + '=' * 90)
    raise RuntimeError('STOPPED ON PURPOSE at the safety gate (this is expected the first time).')
print('Gate passed. Proceeding to download.')

Drive/target volume: total 115.7 GB | free 88.7 GB   ||  local disk free 93.4 GB


,status,bytes,content_type,last_modified,error,body
incremental_book_L2,200,183111008,text/csv,None,None,None
trades,200,27887574,text/csv,None,None,None
book_snapshot_25,200,60541658,text/csv,None,None,None



Total compressed size of the files above: 0.27 GB (files with unknown size: 0)
Gate passed. Proceeding to download.


## 3 · Download (only these files, into a NEW hidden staging folder; renamed to the final folder only if every test passes)

In [ ]:
def download(url, dest, expected_bytes, tries=4):
    part = dest.with_name(dest.name + '.part'); t0 = time.time()
    for attempt in range(1, tries + 1):
        have = part.stat().st_size if part.exists() else 0
        hdr = dict(HEADERS)
        if have: hdr['Range'] = f'bytes={have}-'
        try:
            with requests.get(url, headers=hdr, stream=True, allow_redirects=True, timeout=(30, 120)) as r:
                if r.status_code == 416 and expected_bytes and have == expected_bytes: break
                if r.status_code not in (200, 206): raise RuntimeError(f'HTTP {r.status_code}')
                mode = 'ab' if (r.status_code == 206 and have) else 'wb'; got = have if mode == 'ab' else 0; nxt = 0.05
                with open(part, mode) as f:
                    for chunk in r.iter_content(chunk_size=8 << 20):
                        f.write(chunk); got += len(chunk)
                        if expected_bytes and got / expected_bytes >= nxt: print(f'   {dest.name}: {got/1e6:,.0f} MB ({got/expected_bytes:.0%}) {time.time()-t0:.0f}s'); nxt += 0.05
            break
        except Exception as e:
            print(f'   attempt {attempt} failed: {type(e).__name__}: {str(e)[:120]}'); time.sleep(3 * attempt)
            if attempt == tries: raise
    size = part.stat().st_size
    if expected_bytes and size != expected_bytes: raise RuntimeError(f'size mismatch for {dest.name}: got {size}, server said {expected_bytes}')
    part.rename(dest); return size
RAW = STAGE / 'raw'; RAW.mkdir(parents=True, exist_ok=False); DL = {}
t0 = time.time()
for dt, req in TYPES.items():
    if not (req or INCLUDE_BOOK_SNAPSHOT_25): continue
    dest = RAW / fname_for(dt); print('downloading', dt); size = download(url_for(dt), dest, PROBE[dt]['bytes'])
    DL[dt] = {'file': dest.name, 'bytes': size, 'url': url_for(dt)}
tick('download', t0)
for dt in DL: print(dt, f"{DL[dt]['bytes']/1e6:,.1f} MB"); DL[dt]['path'] = RAW / DL[dt]['file']
REPORT['download'] = {k: {x: y for x, y in v.items() if x != 'path'} for k, v in DL.items()}

downloading incremental_book_L2
   binance_incremental_book_L2_2024-05-01_BTCUSDT.csv.gz: 17 MB (9%) 0s
   binance_incremental_book_L2_2024-05-01_BTCUSDT.csv.gz: 25 MB (14%) 1s
   binance_incremental_book_L2_2024-05-01_BTCUSDT.csv.gz: 34 MB (18%) 1s
   binance_incremental_book_L2_2024-05-01_BTCUSDT.csv.gz: 42 MB (23%) 1s
   binance_incremental_book_L2_2024-05-01_BTCUSDT.csv.gz: 50 MB (27%) 1s
   binance_incremental_book_L2_2024-05-01_BTCUSDT.csv.gz: 59 MB (32%) 1s
   binance_incremental_book_L2_2024-05-01_BTCUSDT.csv.gz: 67 MB (37%) 1s
   binance_incremental_book_L2_2024-05-01_BTCUSDT.csv.gz: 75 MB (41%) 1s
   binance_incremental_book_L2_2024-05-01_BTCUSDT.csv.gz: 84 MB (46%) 1s
   binance_incremental_book_L2_2024-05-01_BTCUSDT.csv.gz: 92 MB (50%) 1s
   binance_incremental_book_L2_2024-05-01_BTCUSDT.csv.gz: 109 MB (60%) 1s
   binance_incremental_book_L2_2024-05-01_BTCUSDT.csv.gz: 117 MB (64%) 1s
   binance_incremental_book_L2_2024-05-01_BTCUSDT.csv.gz: 126 MB (69%) 2s
   binance_increm

## 4 · Integrity: SHA-256, full gzip decompression test, and an independent line count

In [ ]:
t0 = time.time()
for dt, d in DL.items():
    p = d['path']; d['sha256'] = sha256_file(p); n_lines = 0; n_bytes = 0; ok = True
    try:
        with gzip.open(p, 'rb') as f:
            while True:
                b = f.read(32 << 20)
                if not b: break
                n_bytes += len(b); n_lines += b.count(b'\n')
    except Exception as e: ok = False; d['gzip_error'] = str(e)[:200]
    d.update({'gzip_ok': ok, 'uncompressed_bytes': n_bytes, 'lines_incl_header': n_lines})
    print(f"{dt}: gzip_ok={ok} | uncompressed {n_bytes/1e9:.2f} GB | lines {n_lines:,} | sha256 {d['sha256'][:16]}...")
tick('integrity', t0); REPORT['integrity'] = {k: {x: y for x, y in v.items() if x != 'path'} for k, v in DL.items()}

incremental_book_L2: gzip_ok=True | uncompressed 2.28 GB | lines 31,005,312 | sha256 5173777ef4dc7418...
trades: gzip_ok=True | uncompressed 0.20 GB | lines 2,401,095 | sha256 6122f412ca124d1a...
book_snapshot_25: gzip_ok=True | uncompressed 0.67 GB | lines 790,102 | sha256 405f50e99bd224ed...
[integrity] 17.5 s | peak memory so far 0.25 GB


## 5 · Schemas (do the columns match what Tardis documents?)

In [ ]:
EXPECTED = {
 'incremental_book_L2': ['exchange', 'symbol', 'timestamp', 'local_timestamp', 'is_snapshot', 'side', 'price', 'amount'],
 'trades': ['exchange', 'symbol', 'timestamp', 'local_timestamp', 'id', 'side', 'price', 'amount'],
 'book_snapshot_25': ['exchange', 'symbol', 'timestamp', 'local_timestamp'] + [f'{s}[{i}].{k}' for i in range(25) for s in ('asks', 'bids') for k in ('price', 'amount')]}
def col_types(dt):
    cols = EXPECTED[dt]; t = {'exchange': pa.string(), 'symbol': pa.string(), 'timestamp': pa.int64(), 'local_timestamp': pa.int64()}
    if dt == 'incremental_book_L2': t.update({'is_snapshot': pa.bool_(), 'side': pa.string(), 'price': pa.float64(), 'amount': pa.float64()})
    elif dt == 'trades': t.update({'id': pa.string(), 'side': pa.string(), 'price': pa.float64(), 'amount': pa.float64()})
    else: t.update({c: pa.float64() for c in cols[4:]})
    return t
def reader(dt, block_mb=48):
    ro = pacsv.ReadOptions(block_size=block_mb << 20); co = pacsv.ConvertOptions(column_types=col_types(dt), strings_can_be_null=False, true_values=['true', 'True', 'TRUE'], false_values=['false', 'False', 'FALSE'])
    return pacsv.open_csv(DL[dt]['path'], read_options=ro, convert_options=co)
SCHEMA = {}
for dt, d in DL.items():
    with gzip.open(d['path'], 'rt') as f: header = f.readline().strip().split(','); first = f.readline().strip()
    match = header == EXPECTED[dt]; SCHEMA[dt] = {'columns_found': len(header), 'matches_documented_header': match, 'missing': [c for c in EXPECTED[dt] if c not in header], 'extra': [c for c in header if c not in EXPECTED[dt]]}
    print(f'{dt}: {len(header)} columns | matches documented header: {match} | missing {SCHEMA[dt]["missing"][:4]} extra {SCHEMA[dt]["extra"][:4]}')
    if dt != 'book_snapshot_25': print('   header:', ','.join(header))
REPORT['schema'] = SCHEMA
assert all(v['matches_documented_header'] for v in SCHEMA.values()), 'A file does not match the documented header: STOP and paste the output above to Claude (do not guess column meanings).'

incremental_book_L2: 8 columns | matches documented header: True | missing [] extra []
   header: exchange,symbol,timestamp,local_timestamp,is_snapshot,side,price,amount
trades: 8 columns | matches documented header: True | missing [] extra []
   header: exchange,symbol,timestamp,local_timestamp,id,side,price,amount
book_snapshot_25: 104 columns | matches documented header: True | missing [] extra []


## 6 · Timestamps, coverage, snapshot boundaries (one streaming pass over `incremental_book_L2`, then `trades`)

In [ ]:
def vc(col): return {x['values']: x['counts'] for x in pc.value_counts(col).to_pylist()}
DAY0_US = int(pd.Timestamp(DAY, tz='UTC').timestamp() * 1e6); DAY1_US = DAY0_US + 86_400_000_000
GAP_TH = {'>1s': 1e6, '>10s': 1e7, '>60s': 6e7, '>300s': 3e8}
def iso_us(v): return None if v is None else pd.Timestamp(int(v), unit='us', tz='UTC').isoformat()
class Gaps:
    def __init__(s): s.last = None; s.cnt = Counter(); s.top = []
    def add(s, lts):                        # lts: distinct-or-repeated increasing-ish local timestamps (int64)
        x = lts if s.last is None else np.r_[s.last, lts]; d = np.diff(x); s.last = int(lts[-1])
        for k, th in GAP_TH.items(): s.cnt[k] += int((d > th).sum())
        for i in np.flatnonzero(d > 6e7)[:40]: s.top.append((int(d[i]), int(x[i]))); s.top = sorted(s.top, reverse=True)[:10]
def scan_incremental():
    rd = reader('incremental_book_L2'); n = 0; tmin = tmax = lmin = lmax = None; inv_t = inv_l = 0; last_t = last_l = None; snap_rows = 0; pre_rows = 0; seen_snap = False
    side_ct = Counter(); zero_amt = 0; neg = 0; nan = 0; badpx = 0; gaps = Gaps(); batches = []; cur = None; prev_flag = None; hours = Counter(); out_day = 0; lat_b = []; prev_l_for_gap = None; sym_ct = Counter(); ex_ct = Counter()
    for b in rd:
        ts = b.column('timestamp').to_numpy(); ls = b.column('local_timestamp').to_numpy(); fl = b.column('is_snapshot').to_numpy(zero_copy_only=False).astype(bool); amt = b.column('amount').to_numpy(); px = b.column('price').to_numpy()
        isb = pc.equal(b.column('side'), 'bid').to_numpy(zero_copy_only=False).astype(bool)
        m = len(ts); base = n; n += m
        tmin = int(ts.min()) if tmin is None else min(tmin, int(ts.min())); tmax = int(ts.max()) if tmax is None else max(tmax, int(ts.max())); lmin = int(ls.min()) if lmin is None else min(lmin, int(ls.min())); lmax = int(ls.max()) if lmax is None else max(lmax, int(ls.max()))
        xt = ts if last_t is None else np.r_[last_t, ts]; inv_t += int((np.diff(xt) < 0).sum()); last_t = int(ts[-1]); xl = ls if last_l is None else np.r_[last_l, ls]; inv_l += int((np.diff(xl) < 0).sum()); last_l = int(ls[-1])
        for k, v in vc(b.column('side')).items(): side_ct[k] += v
        for k, v in vc(b.column('symbol')).items(): sym_ct[k] += v
        for k, v in vc(b.column('exchange')).items(): ex_ct[k] += v
        lat_b.append((ls - ts)[:: max(1, len(ts) // 20000)]); out_day += int(((ls < DAY0_US) | (ls >= DAY1_US)).sum())
        zero_amt += int((amt == 0).sum()); neg += int((amt < 0).sum()); nan += int(np.isnan(amt).sum() + np.isnan(px).sum()); badpx += int((px <= 0).sum())
        snap_rows += int(fl.sum())
        if not seen_snap:
            if fl.any(): pre_rows += int(np.argmax(fl)); seen_snap = True
            else: pre_rows += m
        chg = np.flatnonzero(fl[1:] != fl[:-1]) + 1 if m > 1 else np.array([], int); starts = [0] + [int(x) for x in chg]; bounds = starts + [m]
        for j_, s0 in enumerate(starts):
            e0 = bounds[j_ + 1]; seg_flag = bool(fl[s0]); cont = (j_ == 0 and prev_flag is not None and seg_flag == prev_flag)
            if seg_flag:
                if cont and cur is not None: cur['rows'] += e0 - s0; cur['bid_rows'] += int(isb[s0:e0].sum()); cur['ask_rows'] += int((~isb[s0:e0]).sum())
                else:
                    if cur is not None: batches.append(cur)
                    cur = {'start_row': base + s0, 'local_ts': int(ls[s0]), 'rows': e0 - s0, 'bid_rows': int(isb[s0:e0].sum()), 'ask_rows': int((~isb[s0:e0]).sum())}
            elif cur is not None: batches.append(cur); cur = None
        prev_flag = bool(fl[-1])
        hh = pd.Series((ls - DAY0_US) // 3_600_000_000).value_counts()
        for k, v in hh.items(): hours[int(k)] += int(v)
        gaps.add(ls)
        if n % 20_000_000 < m: print(f'   scanned {n:,} rows')
    if cur is not None: batches.append(cur)
    lat_b = np.concatenate(lat_b) if lat_b else np.array([0])
    return {'local_outside_day_rows': out_day, 'book_local_minus_exchange_us_median': float(np.median(lat_b)), 'book_local_minus_exchange_us_p99': float(np.percentile(lat_b, 99)), 'rows': n, 'timestamp_min': iso_us(tmin), 'timestamp_max': iso_us(tmax), 'local_min': iso_us(lmin), 'local_max': iso_us(lmax), 'ts_inversions': inv_t, 'local_inversions': inv_l, 'snapshot_rows': snap_rows, 'rows_before_first_snapshot': pre_rows, 'sides': dict(side_ct), 'symbols': dict(sym_ct), 'exchanges': dict(ex_ct),
            'amount_zero_rows': zero_amt, 'amount_negative': neg, 'nan_cells': nan, 'price_le_zero': badpx, 'snapshot_batches': batches, 'rows_per_utc_hour': {int(k): int(v) for k, v in sorted(hours.items())}, 'gap_counts': dict(gaps.cnt), 'largest_gaps_s': [(round(g / 1e6, 1), iso_us(t)) for g, t in gaps.top], 'timestamp_unit_check_ok': bool(tmin and 1.0e15 < tmin < 2.0e15)}
t0 = time.time(); SC = scan_incremental(); tick('scan_incremental', t0); REPORT['book_scan'] = SC
print(json.dumps({k: v for k, v in SC.items() if k not in ('snapshot_batches', 'rows_per_utc_hour')}, indent=1, default=jsonable))
print('snapshot batches:', len(SC['snapshot_batches'])); display(pd.DataFrame(SC['snapshot_batches']).assign(time_utc=lambda d: d.local_ts.map(iso_us)).head(30))
print('rows per UTC hour:', SC['rows_per_utc_hour'])

   scanned 20,497,640 rows
[scan_incremental] 33.3 s | peak memory so far 2.08 GB
{
 "local_outside_day_rows": 0,
 "book_local_minus_exchange_us_median": 3922.0,
 "book_local_minus_exchange_us_p99": 8111.0,
 "rows": 31005311,
 "timestamp_min": "2024-05-01T00:00:01.419267+00:00",
 "timestamp_max": "2024-05-01T23:59:59.946000+00:00",
 "local_min": "2024-05-01T00:00:01.419267+00:00",
 "local_max": "2024-05-01T23:59:59.949379+00:00",
 "ts_inversions": 0,
 "local_inversions": 0,
 "snapshot_rows": 2045,
 "rows_before_first_snapshot": 0,
 "sides": {
  "ask": 14955989,
  "bid": 16049322
 },
 "symbols": {
  "BTCUSDT": 31005311
 },
 "exchanges": {
  "binance": 31005311
 },
 "amount_zero_rows": 14870451,
 "amount_negative": 0,
 "nan_cells": 0,
 "price_le_zero": 0,
 "gap_counts": {
  ">1s": 0,
  ">10s": 0,
  ">60s": 0,
  ">300s": 0
 },
 "largest_gaps_s": [],
 "timestamp_unit_check_ok": true
}
snapshot batches: 1


,start_row,local_ts,rows,bid_rows,ask_rows,time_utc
0,0,1714521601419267,2045,1029,1016,2024-05-01T00:00:01.419267+00:00


rows per UTC hour: {0: 953192, 1: 995865, 2: 971546, 3: 561283, 4: 483774, 5: 623055, 6: 882702, 7: 2711718, 8: 2278414, 9: 1325496, 10: 1223043, 11: 852095, 12: 1216329, 13: 1364501, 14: 1377278, 15: 1474123, 16: 1334028, 17: 1288752, 18: 2722860, 19: 2525960, 20: 1287159, 21: 919725, 22: 869351, 23: 763062}


In [ ]:
def scan_trades():
    rd = reader('trades'); n = 0; tmin = tmax = lmin = lmax = None; inv = 0; last = None; side_ct = Counter(); ids_first = ids_last = None; id_num = True; px_min, px_max, q_min, q_max = 1e18, -1e18, 1e18, -1e18; lat = []; out_day = 0; gaps = Gaps(); id_gap = 0; id_dup = 0; prev_id = None
    for b in rd:
        ts = b.column('timestamp').to_numpy(); ls = b.column('local_timestamp').to_numpy(); px = b.column('price').to_numpy(); q = b.column('amount').to_numpy(); n += len(ts)
        tmin = int(ts.min()) if tmin is None else min(tmin, int(ts.min())); tmax = int(ts.max()) if tmax is None else max(tmax, int(ts.max())); lmin = int(ls.min()) if lmin is None else min(lmin, int(ls.min())); lmax = int(ls.max()) if lmax is None else max(lmax, int(ls.max()))
        x = ls if last is None else np.r_[last, ls]; inv += int((np.diff(x) < 0).sum()); last = int(ls[-1]); px_min, px_max, q_min, q_max = min(px_min, float(px.min())), max(px_max, float(px.max())), min(q_min, float(q.min())), max(q_max, float(q.max()))
        for k_, v_ in vc(b.column('side')).items(): side_ct[k_] += v_
        lat.append((ls - ts)[:: max(1, len(ts) // 20000)]); gaps.add(ls); out_day += int(((ls < DAY0_US) | (ls >= DAY1_US)).sum())
        try:
            ids = pc.cast(b.column('id'), pa.int64()).to_numpy(zero_copy_only=False); xi = ids if prev_id is None else np.r_[prev_id, ids]; d = np.diff(xi); id_gap += int((d > 1).sum()); id_dup += int((d <= 0).sum()); prev_id = int(ids[-1])
        except Exception: id_num = False
    lat = np.concatenate(lat) if lat else np.array([0])
    return {'local_outside_day_rows': out_day, 'amount_min': q_min, 'rows': n, 'timestamp_min': iso_us(tmin), 'timestamp_max': iso_us(tmax), 'local_min': iso_us(lmin), 'local_max': iso_us(lmax), 'local_inversions': inv, 'sides': dict(side_ct), 'price_range': [px_min, px_max], 'amount_range': [q_min, q_max], 'id_numeric': id_num, 'id_gaps(>1)': id_gap, 'id_dup_or_backwards': id_dup,
            'local_minus_exchange_us_median': float(np.median(lat)), 'local_minus_exchange_us_p99': float(np.percentile(lat, 99)), 'gap_counts': dict(gaps.cnt), 'largest_gaps_s': [(round(g / 1e6, 1), iso_us(t)) for g, t in gaps.top]}
t0 = time.time(); TR = scan_trades(); tick('scan_trades', t0); REPORT['trades_scan'] = TR; print(json.dumps(TR, indent=1, default=jsonable))

[scan_trades] 2.4 s | peak memory so far 2.08 GB
{
 "local_outside_day_rows": 0,
 "amount_min": 1e-05,
 "rows": 2401094,
 "timestamp_min": "2024-04-30T23:59:59.996000+00:00",
 "timestamp_max": "2024-05-01T23:59:59.998000+00:00",
 "local_min": "2024-05-01T00:00:00.000132+00:00",
 "local_max": "2024-05-01T23:59:59.999712+00:00",
 "local_inversions": 0,
 "sides": {
  "sell": 1258548,
  "buy": 1142546
 },
 "price_range": [
  56552.82,
  60841.63
 ],
 "amount_range": [
  1e-05,
  29.14457
 ],
 "id_numeric": true,
 "id_gaps(>1)": 0,
 "id_dup_or_backwards": 0,
 "local_minus_exchange_us_median": 4151.0,
 "local_minus_exchange_us_p99": 256007.93000000145,
 "gap_counts": {
  ">1s": 5381,
  ">10s": 0,
  ">60s": 0,
  ">300s": 0
 },
 "largest_gaps_s": []
}


## 7 · Rebuild the order book and test it (no graphs, no models)

How the replay works (Tardis rules): skip rows before the first `is_snapshot` batch; when a snapshot batch starts, clear the book; `amount` is the *new total* at that price (not a delta); `0` removes the level. A **message** = consecutive rows with the same `local_timestamp`; the book is only inspected **after a completed message** (never in the middle of one). I replay two variants at once: **unpruned** (levels are only removed when set to 0) and **pruned to 1000 levels per side** (the Hugging Face README warns that unpruned books can keep stale "ghost" levels).

In [ ]:
class Book:
    def __init__(s, K): s.bids = {}; s.asks = {}; s.bh = []; s.ah = []; s.K = K
    def reset(s): s.bids.clear(); s.asks.clear(); s.bh.clear(); s.ah.clear()
    def upd(s, is_bid, p, a):
        if is_bid:
            d = s.bids
            if a > 0:
                if p not in d: heapq.heappush(s.bh, -p)
                d[p] = a
            else: d.pop(p, None)
        else:
            d = s.asks
            if a > 0:
                if p not in d: heapq.heappush(s.ah, p)
                d[p] = a
            else: d.pop(p, None)
    def best(s):
        bh = s.bh; bd = s.bids
        while bh and -bh[0] not in bd: heapq.heappop(bh)
        ah = s.ah; ad = s.asks
        while ah and ah[0] not in ad: heapq.heappop(ah)
        if len(bh) > 4 * len(bd) + 4000: s.bh = bh = [-p for p in bd]; heapq.heapify(bh)
        if len(ah) > 4 * len(ad) + 4000: s.ah = ah = list(ad); heapq.heapify(ah)
        return (-bh[0] if bh else float('nan')), (ah[0] if ah else float('nan'))
    def prune(s):
        K = s.K
        if K and len(s.bids) > K + 50:
            for p in sorted(s.bids)[:len(s.bids) - K]: del s.bids[p]
        if K and len(s.asks) > K + 50:
            for p in sorted(s.asks)[K:]: del s.asks[p]
    def top(s, n=25):
        b = heapq.nlargest(n, s.bids.items()); a = heapq.nsmallest(n, s.asks.items()); return b, a
    def fingerprint(s): return hashlib.sha256(json.dumps([sorted(s.bids.items())[-50:], sorted(s.asks.items())[:50], len(s.bids), len(s.asks)]).encode()).hexdigest()[:16]

S25_META = {}
def load_snapshot25_sample(stride=2000):
    """sample every `stride`-th row of Tardis' own book_snapshot_25 (used only as a CROSS-CHECK, keyed by local_timestamp)"""
    if 'book_snapshot_25' not in DL: return {}, 0
    rd = reader('book_snapshot_25', block_mb=32); samp = {}; n = 0; S25_META['rows_sampled'] = 0
    for b in rd:
        m = b.num_rows; idx = [i for i in range(m) if (n + i) % stride == 0]; n += m
        if not idx: continue
        sub = b.take(pa.array(idx)); L = sub.column('local_timestamp').to_numpy()
        ap = np.column_stack([np.asarray(sub.column(f'asks[{i}].price').to_numpy(zero_copy_only=False), float) for i in range(25)]); aa = np.column_stack([np.asarray(sub.column(f'asks[{i}].amount').to_numpy(zero_copy_only=False), float) for i in range(25)])
        bp = np.column_stack([np.asarray(sub.column(f'bids[{i}].price').to_numpy(zero_copy_only=False), float) for i in range(25)]); ba = np.column_stack([np.asarray(sub.column(f'bids[{i}].amount').to_numpy(zero_copy_only=False), float) for i in range(25)])
        S25_META['rows_sampled'] += len(L)
        for j in range(len(L)): samp[int(L[j])] = (ap[j], aa[j], bp[j], ba[j])
    return samp, n
TOPNS = (5, 10, 25)
def compare_top(book, snap):
    """compare the rebuilt book with ONE book_snapshot_25 row (both taken after the same completed message)"""
    ap, aa, bp, ba = snap; b, a = book.top(25); r = {}
    for N in TOPNS: r[N] = {'lv': 0, 'p': 0, 'q': 0, 'all_p': True, 'all_pq': True}
    for sp_, sa_, mine in ((bp, ba, b), (ap, aa, a)):
        for i in range(25):
            sp = sp_[i]
            if not np.isfinite(sp): continue
            okp = i < len(mine) and mine[i][0] == sp; okq = okp and math.isclose(mine[i][1], sa_[i], rel_tol=1e-9, abs_tol=1e-12)
            for N in TOPNS:
                if i < N:
                    d = r[N]; d['lv'] += 1; d['p'] += int(okp); d['q'] += int(okq)
                    if not okp: d['all_p'] = False
                    if not okq: d['all_pq'] = False
    bb_s = bp[0]; ba_s = ap[0]; bb_m = b[0][0] if b else float('nan'); ba_m = a[0][0] if a else float('nan')
    r['best_ok'] = bool(np.isfinite(bb_s) and np.isfinite(ba_s) and bb_m == bb_s and ba_m == ba_s)
    r['spread_diff_bps'] = float(abs((ba_m - bb_m) - (ba_s - bb_s)) / ((ba_s + bb_s) / 2) * 1e4) if np.isfinite(bb_s) and np.isfinite(ba_s) and np.isfinite(bb_m) and np.isfinite(ba_m) else float('nan')
    r['spread_equal'] = bool(r['spread_diff_bps'] == r['spread_diff_bps'] and r['spread_diff_bps'] < 1e-9)
    return r
def new_cmp(): return {'checkpoints': 0, 'excluded_crossed_or_locked': 0, 'lv': Counter(), 'p': Counter(), 'q': Counter(), 'all_p': Counter(), 'all_pq': Counter(), 'best_ok': 0, 'spread_equal': 0, 'spread_diff': [], 'by_hour': Counter(), 'by_hour_ok': Counter(), 'empty_book_skipped': 0}
def depth_stats(bk, mid):
    d10 = d25 = 0.0; far = 0; lo10, lo25, lo_far = mid * 0.999, mid * 0.9975, mid * 0.95; hi10, hi25, hi_far = mid * 1.001, mid * 1.0025, mid * 1.05
    for p, q in bk.bids.items():
        if p >= lo10: d10 += q
        if p >= lo25: d25 += q
        if p < lo_far: far += 1
    for p, q in bk.asks.items():
        if p <= hi10: d10 += q
        if p <= hi25: d25 += q
        if p > hi_far: far += 1
    return d10, d25, far
def tick_of(bk):
    ps = np.array(sorted(bk.bids)[-200:] + sorted(bk.asks)[:200]); d = np.diff(ps); d = np.round(d[d > 1e-9], 8)
    return float(d.min()) if len(d) else float('nan')
def run_replay(max_rows=None, run_unpruned=True, samp=None, collect=True, K=1000, verbose=True):
    rd = reader('incremental_book_L2'); variants = {'pruned1000': Book(K)}
    if run_unpruned: variants['unpruned'] = Book(0)
    V = list(variants.items()); TL = {k: {'lts': array('q'), 'bb': array('d'), 'ba': array('d')} for k in variants}
    st = {'rows': 0, 'skipped_pre': 0, 'messages': 0, 'snap_batches': 0, 'checks': 0, 'locked': Counter(), 'crossed': Counter(), 'max_cross_run': Counter(), 'run': Counter(), 'resets': 0}
    cmp = {k: new_cmp() for k in variants}
    depth = {k: [] for k in variants}; next_sample = [None]; started = False; pL = None; pf = None; t0 = time.time(); done_rows = 0; snap_checks = []
    def end_msg(L, was_snap):
        st['messages'] += 1; st['checks'] += 1
        if was_snap: st['snap_batches'] += 1
        for k, bk in V:
            if bk.K: bk.prune()
            bb, ba = bk.best()
            if collect: tl = TL[k]; tl['lts'].append(L); tl['bb'].append(bb); tl['ba'].append(ba)
            if bb == bb and ba == ba:
                if bb > ba: st['crossed'][k] += 1; st['run'][k] += 1; st['max_cross_run'][k] = max(st['max_cross_run'][k], st['run'][k])
                elif bb == ba: st['locked'][k] += 1; st['run'][k] = 0
                else: st['run'][k] = 0
            if samp is not None and L in samp:
                c = cmp[k]
                if not (bb == bb and ba == ba): c['empty_book_skipped'] += 1
                elif bb >= ba: c['excluded_crossed_or_locked'] += 1       # Tardis removes overlapping levels from its snapshots, so a crossed/locked moment is not comparable
                else:
                    r = compare_top(bk, samp[L]); c['checkpoints'] += 1; hr = int((L - DAY0_US) // 3_600_000_000)
                    for N in TOPNS:
                        d = r[N]; c['lv'][N] += d['lv']; c['p'][N] += d['p']; c['q'][N] += d['q']; c['all_p'][N] += int(d['all_p']); c['all_pq'][N] += int(d['all_pq'])
                    c['best_ok'] += int(r['best_ok']); c['spread_equal'] += int(r['spread_equal']); c['spread_diff'].append(r['spread_diff_bps']); c['by_hour'][hr] += 1; c['by_hour_ok'][hr] += int(r[25]['all_pq'])
        if collect and (next_sample[0] is None or L >= next_sample[0]):
            next_sample[0] = L + 30_000_000
            for k, bk in V:
                bb, ba = bk.best()
                if bb == bb and ba == ba:
                    mid = (bb + ba) / 2; depth[k].append((L, mid, (ba - bb) / mid * 1e4, len(bk.bids), len(bk.asks)) + depth_stats(bk, mid))
    for b in rd:
        Ls = b.column('local_timestamp').to_numpy().tolist(); Fs = b.column('is_snapshot').to_numpy(zero_copy_only=False).astype(bool).tolist(); Bs = pc.equal(b.column('side'), 'bid').to_numpy(zero_copy_only=False).astype(bool).tolist(); Ps = b.column('price').to_numpy().tolist(); As = b.column('amount').to_numpy().tolist()
        for i in range(len(Ls)):
            L = Ls[i]; f = Fs[i]
            if max_rows is not None and st['rows'] >= max_rows: break
            st['rows'] += 1
            if started and (f != pf or ((not f) and L != pL)): end_msg(pL, pf)
            if not started:
                if not f: st['skipped_pre'] += 1; pL = L; pf = f; continue
                started = True
            if f and (pf is None or not pf or pf is False):
                for _, bk in V: bk.reset()
                st['resets'] += 1
            for _, bk in V: bk.upd(Bs[i], Ps[i], As[i])
            pL = L; pf = f
        else:
            if verbose and st['rows'] // 5_000_000 != done_rows // 5_000_000:
                el = time.time() - t0; print(f"   replayed {st['rows']:,} rows | {el:.0f}s | messages {st['messages']:,}")
            done_rows = st['rows']; continue
        break
    if started: end_msg(pL, pf)
    out = {'stats': st, 'cmp': cmp, 'timeline': TL, 'depth': depth, 'tick_est': {k: tick_of(bk) for k, bk in V}, 'fp': {k: bk.fingerprint() for k, bk in V}, 'final_levels': {k: (len(bk.bids), len(bk.asks)) for k, bk in V}, 'seconds': time.time() - t0}
    return out
QUICK_TEST_ROWS = None      # leave None for the full day. For a quick dry run on a small file set e.g. 2_000_000.
S25_STRIDE = max(1, (DL['book_snapshot_25']['lines_incl_header'] - 1) // 20000) if 'book_snapshot_25' in DL else 1
t0 = time.time(); SAMP, N_S25 = load_snapshot25_sample(S25_STRIDE); tick('load_snapshot25_sample', t0); print(f'book_snapshot_25 rows: {N_S25:,} | every {S25_STRIDE}-th row sampled -> {len(SAMP):,} distinct local_timestamps to compare')
t0 = time.time(); RP = run_replay(max_rows=QUICK_TEST_ROWS, run_unpruned=True, samp=SAMP, collect=True); tick('replay', t0)
st = RP['stats']; print({k: (dict(v) if isinstance(v, Counter) else v) for k, v in st.items()}); print('final levels (bids, asks):', RP['final_levels'], '| estimated tick size:', RP['tick_est'])

[load_snapshot25_sample] 16.9 s | peak memory so far 2.08 GB
book_snapshot_25 rows: 790,101 | every 39-th row sampled -> 20,259 distinct local_timestamps to compare
   replayed 5,457,704 rows | 59s | messages 250,025
   replayed 10,247,801 rows | 100s | messages 317,006
   replayed 15,030,188 rows | 187s | messages 463,980
   replayed 20,497,640 rows | 275s | messages 605,190
   replayed 25,286,884 rows | 335s | messages 690,822
   replayed 30,074,828 rows | 417s | messages 817,202
[replay] 443.5 s | peak memory so far 2.52 GB
{'rows': 31005311, 'skipped_pre': 0, 'messages': 861138, 'snap_batches': 1, 'checks': 861138, 'locked': {}, 'crossed': {}, 'max_cross_run': {}, 'run': {'pruned1000': 0, 'unpruned': 0}, 'resets': 1}
final levels (bids, asks): {'pruned1000': (982, 827), 'unpruned': (26337, 31048)} | estimated tick size: {'pruned1000': 0.01, 'unpruned': 0.01}


## 8 · Order-book reconstruction results (measurements only; the verdict comes later and is kept separate from the code tests)

**Alignment rule for the `book_snapshot_25` comparison.** A Tardis `book_snapshot_25` row carries the `local_timestamp` of the message that changed the top 25 levels. I therefore compare it with my rebuilt book **only after the completed message whose `local_timestamp` is exactly equal**. Rows with no such message are counted as *unmatched* (never forced onto a neighbouring message). Moments where my book is empty, crossed or locked are *excluded* and counted (Tardis removes overlapping levels from its snapshots, so those moments are not comparable). What remains are the **aligned snapshot checkpoints**. Both datasets come from the same provider, so agreement is a *consistency* check, not independent validation.

In [ ]:
MSG = {k: np.frombuffer(v['lts'], dtype=np.int64) for k, v in RP['timeline'].items()}
n_msg = RP['stats']['messages']
CROSS = {}
for k in RP['timeline']:
    c = RP['stats']['crossed'][k]; l = RP['stats']['locked'][k]; CROSS[k] = {'messages_checked': n_msg, 'crossed_messages': c, 'locked_messages': l, 'crossed_share': c / max(n_msg, 1), 'locked_share': l / max(n_msg, 1), 'crossed_plus_locked_share': (c + l) / max(n_msg, 1), 'max_consecutive_crossed': RP['stats']['max_cross_run'][k]}
display(pd.DataFrame(CROSS).T); REPORT['crossed'] = CROSS
def sh(a, b): return (a / b) if b else None
CMP = {}
for k, c in RP['cmp'].items():
    matched = c['checkpoints'] + c['excluded_crossed_or_locked'] + c['empty_book_skipped']; sd = np.array(c['spread_diff'], float) if c['spread_diff'] else np.array([np.nan])
    d = {'snapshot25_rows_total': (N_S25 if SAMP else 0), 'snapshot25_rows_sampled': S25_META.get('rows_sampled'), 'distinct_local_timestamps_sampled': len(SAMP), 'matched_to_a_completed_message': matched, 'unmatched_no_message_with_equal_local_timestamp': len(SAMP) - matched,
         'excluded_book_crossed_or_locked': c['excluded_crossed_or_locked'], 'excluded_book_empty': c['empty_book_skipped'], 'ALIGNED_CHECKPOINTS': c['checkpoints']}
    for N in TOPNS:
        d[f'top{N}_price_agreement_(levels)'] = sh(c['p'][N], c['lv'][N]); d[f'top{N}_quantity_agreement_(price AND qty equal)'] = sh(c['q'][N], c['lv'][N]); d[f'top{N}_quantity_given_price_match'] = sh(c['q'][N], c['p'][N]); d[f'top{N}_checkpoints_all_prices_exact'] = sh(c['all_p'][N], c['checkpoints']); d[f'top{N}_checkpoints_all_prices_and_qty_exact'] = sh(c['all_pq'][N], c['checkpoints'])
    d.update({'best_bid_and_ask_both_equal': sh(c['best_ok'], c['checkpoints']), 'spread_exactly_equal': sh(c['spread_equal'], c['checkpoints']), 'spread_abs_diff_bps_median': float(np.nanmedian(sd)), 'spread_abs_diff_bps_p99': float(np.nanpercentile(sd, 99)),
              'worst_hours_(share_of_checkpoints_all25_exact)': sorted([(h, round(c['by_hour_ok'][h] / c['by_hour'][h], 3), c['by_hour'][h]) for h in c['by_hour']], key=lambda x: x[1])[:5]})
    CMP[k] = d
display(pd.DataFrame(CMP).T.T); REPORT['cmp_snapshot25'] = CMP
cov = {}
L_ = MSG['pruned1000']; mins = np.unique((L_ - DAY0_US) // 60_000_000); mins = mins[(mins >= 0) & (mins < 1440)]; d_ = np.diff(L_) if len(L_) > 1 else np.array([0]); big = np.sort(d_)[-8:][::-1]
cov = {'messages': int(len(L_)), 'minutes_with_a_message_of_1440': int(len(mins)), 'coverage_of_day_by_minutes': len(mins) / 1440, 'first_message_utc': iso_us(L_.min()) if len(L_) else None, 'last_message_utc': iso_us(L_.max()) if len(L_) else None,
       'messages_per_second_mean': len(L_) / max((L_.max() - L_.min()) / 1e6, 1) if len(L_) else None, 'gaps_between_messages_over': {kk: int((d_ > th).sum()) for kk, th in GAP_TH.items()}, 'largest_message_gaps_s': [round(float(x) / 1e6, 1) for x in big]}
print(json.dumps(cov, indent=1, default=jsonable)); REPORT['coverage'] = cov
DEPTH = {}
for k, rows in RP['depth'].items():
    if not rows: continue
    df = pd.DataFrame(rows, columns=['L', 'mid', 'spread_bps', 'n_bid_levels', 'n_ask_levels', 'depth10bps', 'depth25bps', 'levels_far_5pct']); df['hour'] = ((df.L - DAY0_US) // 3_600_000_000).astype(int); DEPTH[k] = df
REPORT['depth_series_summary'] = {k: {'spread_bps_median': float(v.spread_bps.median()), 'levels_start': [int(v.n_bid_levels.iloc[0]), int(v.n_ask_levels.iloc[0])], 'levels_end': [int(v.n_bid_levels.iloc[-1]), int(v.n_ask_levels.iloc[-1])], 'levels_max': [int(v.n_bid_levels.max()), int(v.n_ask_levels.max())], 'far_levels_max_(>5pct_from_mid)': int(v.levels_far_5pct.max()), 'depth10bps_median_btc': float(v.depth10bps.median()), 'depth25bps_median_btc': float(v.depth25bps.median()), 'samples': len(v)} for k, v in DEPTH.items()}
print(json.dumps(REPORT['depth_series_summary'], indent=1, default=jsonable))

,messages_checked,crossed_messages,locked_messages,crossed_share,locked_share,crossed_plus_locked_share,max_consecutive_crossed
pruned1000,861138.0,0.0,0.0,0.0,0.0,0.0,0.0
unpruned,861138.0,0.0,0.0,0.0,0.0,0.0,0.0


,pruned1000,unpruned
snapshot25_rows_total,790101,790101
snapshot25_rows_sampled,20259,20259
distinct_local_timestamps_sampled,20259,20259
matched_to_a_completed_message,20259,20259
unmatched_no_message_with_equal_local_timestamp,0,0
excluded_book_crossed_or_locked,0,0
excluded_book_empty,0,0
ALIGNED_CHECKPOINTS,20259,20259
top5_price_agreement_(levels),0.995908,1.0
top5_quantity_agreement_(price AND qty equal),0.995908,1.0


{
 "messages": 861138,
 "minutes_with_a_message_of_1440": 1440,
 "coverage_of_day_by_minutes": 1.0,
 "first_message_utc": "2024-05-01T00:00:01.419267+00:00",
 "last_message_utc": "2024-05-01T23:59:59.949379+00:00",
 "messages_per_second_mean": 9.967044565268543,
 "gaps_between_messages_over": {
  ">1s": 0,
  ">10s": 0,
  ">60s": 0,
  ">300s": 0
 },
 "largest_message_gaps_s": [
  0.4,
  0.4,
  0.4,
  0.3,
  0.3,
  0.3,
  0.3,
  0.3
 ]
}
{
 "pruned1000": {
  "spread_bps_median": 0.0017285019517876328,
  "levels_start": [
   1011,
   1004
  ],
  "levels_end": [
   977,
   743
  ],
  "levels_max": [
   1050,
   1050
  ],
  "far_levels_max_(>5pct_from_mid)": 289,
  "depth10bps_median_btc": 102.11499000000018,
  "depth25bps_median_btc": 216.0915500000008,
  "samples": 2879
 },
 "unpruned": {
  "spread_bps_median": 0.001728494183753152,
  "levels_start": [
   1011,
   1004
  ],
  "levels_end": [
   26340,
   30970
  ],
  "levels_max": [
   29195,
   35743
  ],
  "far_levels_max_(>5pct_from_mi

## 9 · Trade prices versus the rebuilt book — a DIAGNOSTIC (it never passes or fails the reconstruction)

**Why this is only a diagnostic.** Trades legitimately print *at* the best bid or best ask (a sell-taker hits the bid, a buy-taker lifts the ask), and a trade can also print slightly away from my reconstructed touch simply because the **trade feed and the depth feed are separate WebSocket connections** whose messages arrive with different latencies. So a share below 100 % "inside the spread" does not by itself mean the book is wrong.

**Timestamp alignment and tolerance used here.**
* Both feeds are placed on the collector's arrival clock (`local_timestamp`, microseconds) — the same clock domain on the Tardis side (assumption: not verifiable from the files). The exchange `timestamp` is not used for the alignment.
* **Strict (causal) state:** the book after the last completed message strictly **before** the trade's `local_timestamp`. A second variant uses the last message at-or-before.
* **Latency tolerance:** the trade is also tested against the *envelope* of book states over the preceding **100 ms** and **1 s** (lowest best-bid and highest best-ask seen in that window), which absorbs feed-latency differences.
* **Price tolerance:** "at / near" means exactly equal or within **one tick**; the tick size is estimated from the rebuilt book's price grid (printed below), not assumed.
* Classes are mutually exclusive: at best bid · near best bid (≤ 1 tick) · at best ask · near best ask · interior of the spread · outside the spread by more than one tick.

In [ ]:
TICK = RP['tick_est'].get('pruned1000'); TICK = TICK if TICK == TICK and TICK > 0 else 0.01; print('estimated tick size from rebuilt book:', TICK)
def classify(p, bb, ba, tick):
    eps = 1e-9; t = tick * 1.0001
    exact_bid = np.abs(p - bb) < eps; exact_ask = np.abs(p - ba) < eps; near_bid = ~exact_bid & (np.abs(p - bb) <= t); near_ask = ~exact_ask & (np.abs(p - ba) <= t) & ~near_bid & ~exact_bid
    interior = ~exact_bid & ~exact_ask & ~near_bid & ~near_ask & (p > bb) & (p < ba); outside = ~(exact_bid | exact_ask | near_bid | near_ask | interior)
    return {'at_best_bid': exact_bid & ~exact_ask, 'near_best_bid_(<=1tick)': near_bid, 'at_best_ask': exact_ask, 'near_best_ask_(<=1tick)': near_ask & ~exact_ask, 'interior_of_spread': interior, 'outside_spread_(>1tick)': outside}
def trade_diag(book_key, max_trades=300_000):
    rd = reader('trades'); L = MSG[book_key]; BB = np.frombuffer(RP['timeline'][book_key]['bb'], dtype=np.float64); BA = np.frombuffer(RP['timeline'][book_key]['ba'], dtype=np.float64)
    stride = max(1, TR['rows'] // max_trades); tot = Counter(); sellc = Counter(); buyc = Counter(); env = {0.1: Counter(), 1.0: Counter()}; n_valid = 0; n_total = 0; n_noprior = 0; nsell = nbuy = 0; sell_at_ask = buy_at_bid = 0; at_state = Counter(); pos = 0
    for b in rd:
        m = b.num_rows; idx = np.arange((-pos) % stride, m, stride); pos += m
        if len(idx) == 0: continue
        ls = b.column('local_timestamp').to_numpy()[idx]; px = b.column('price').to_numpy()[idx]; sd = pc.equal(b.column('side'), 'sell').to_numpy(zero_copy_only=False).astype(bool)[idx]; unk = pc.equal(b.column('side'), 'unknown').to_numpy(zero_copy_only=False).astype(bool)[idx]
        n_total += len(ls); i_b = np.searchsorted(L, ls, side='left') - 1; ok = i_b >= 0; n_noprior += int((~ok).sum())
        bb = BB[np.where(ok, i_b, 0)]; ba = BA[np.where(ok, i_b, 0)]; valid = ok & np.isfinite(bb) & np.isfinite(ba) & (bb < ba)
        n_valid += int(valid.sum()); cl = classify(px, bb, ba, TICK)
        for name, mask in cl.items():
            mk = mask & valid; tot[name] += int(mk.sum()); sellc[name] += int((mk & sd).sum()); buyc[name] += int((mk & ~sd & ~unk).sum())
        nsell += int((valid & sd).sum()); nbuy += int((valid & ~sd & ~unk).sum())
        sell_at_ask += int((valid & sd & (cl['at_best_ask'] | cl['near_best_ask_(<=1tick)'])).sum()); buy_at_bid += int((valid & ~sd & ~unk & (cl['at_best_bid'] | cl['near_best_bid_(<=1tick)'])).sum())
        i_a = np.searchsorted(L, ls, side='right') - 1; oka = i_a >= 0; ba_a = BA[np.where(oka, i_a, 0)]; bb_a = BB[np.where(oka, i_a, 0)]; va = oka & np.isfinite(bb_a) & np.isfinite(ba_a) & (bb_a < ba_a); ca = classify(px, bb_a, ba_a, TICK)
        at_state['valid'] += int(va.sum()); at_state['outside'] += int((ca['outside_spread_(>1tick)'] & va).sum())
        for dlt in env:
            lo = np.searchsorted(L, ls - int(dlt * 1e6), side='left') - 1; lo = np.maximum(lo, 0); hi = np.maximum(i_b, 0)
            for j in range(len(ls)):
                if not valid[j]: continue
                sb = BB[lo[j]:hi[j] + 1]; sa = BA[lo[j]:hi[j] + 1]; sb = sb[np.isfinite(sb)]; sa = sa[np.isfinite(sa)]
                if len(sb) == 0 or len(sa) == 0: continue
                env[dlt]['valid'] += 1; env[dlt]['inside_envelope_(+-1tick)'] += int(px[j] >= sb.min() - TICK * 1.0001 and px[j] <= sa.max() + TICK * 1.0001)
    v = max(n_valid, 1); out = {'book_variant': book_key, 'trades_sampled': n_total, 'sampling_stride': stride, 'trades_without_prior_book_state': n_noprior, 'trades_with_usable_state': n_valid, 'tick_used': TICK,
           'classification_state_before_trade_(share_of_usable)': {k: tot[k] / v for k in tot}, 'by_taker_side_(share_within_side)': {'sell_taker': {k: sellc[k] / max(nsell, 1) for k in sellc}, 'buy_taker': {k: buyc[k] / max(nbuy, 1) for k in buyc}},
           'within_spread_inclusive_of_touch_(<=1tick)': 1 - tot['outside_spread_(>1tick)'] / v, 'outside_spread_(>1tick)_state_before': tot['outside_spread_(>1tick)'] / v, 'outside_spread_(>1tick)_state_at_or_before': at_state['outside'] / max(at_state['valid'], 1),
           'sell_taker_printing_at_or_near_ask_(unexpected_side)': sell_at_ask / max(nsell, 1), 'buy_taker_printing_at_or_near_bid_(unexpected_side)': buy_at_bid / max(nbuy, 1),
           'latency_tolerant_inside_envelope': {f'last_{int(d*1000)}ms': env[d]['inside_envelope_(+-1tick)'] / max(env[d]['valid'], 1) for d in env}}
    return out
t0 = time.time(); TD = {k: trade_diag(k) for k in (['pruned1000'] + (['unpruned'] if 'unpruned' in MSG else []))}; tick('trade_diagnostic', t0)
TS_COMPAT = {'book_local_range': [SC['local_min'], SC['local_max']], 'trade_local_range': [TR['local_min'], TR['local_max']], 'book_exchange_ts_range': [SC['timestamp_min'], SC['timestamp_max']], 'trade_exchange_ts_range': [TR['timestamp_min'], TR['timestamp_max']], 'both_microseconds': bool(SC['timestamp_unit_check_ok']),
             'book_latency_us_median(local-exchange)': SC['book_local_minus_exchange_us_median'], 'trade_latency_us_median': TR['local_minus_exchange_us_median'], 'latency_offset_trade_minus_book_us_median': TR['local_minus_exchange_us_median'] - SC['book_local_minus_exchange_us_median'], 'book_latency_us_p99': SC['book_local_minus_exchange_us_p99'], 'trade_latency_us_p99': TR['local_minus_exchange_us_p99']}
print(json.dumps({'timestamp_compatibility': TS_COMPAT, 'trade_diagnostic': TD}, indent=1, default=jsonable)); REPORT['trade_book_diagnostic'] = {'timestamps': TS_COMPAT, 'by_variant': TD, 'status': 'DIAGNOSTIC ONLY - not a pass/fail criterion'}

estimated tick size from rebuilt book: 0.01
[trade_diagnostic] 16.9 s | peak memory so far 2.52 GB
{
 "timestamp_compatibility": {
  "book_local_range": [
   "2024-05-01T00:00:01.419267+00:00",
   "2024-05-01T23:59:59.949379+00:00"
  ],
  "trade_local_range": [
   "2024-05-01T00:00:00.000132+00:00",
   "2024-05-01T23:59:59.999712+00:00"
  ],
  "book_exchange_ts_range": [
   "2024-05-01T00:00:01.419267+00:00",
   "2024-05-01T23:59:59.946000+00:00"
  ],
  "trade_exchange_ts_range": [
   "2024-04-30T23:59:59.996000+00:00",
   "2024-05-01T23:59:59.998000+00:00"
  ],
  "both_microseconds": true,
  "book_latency_us_median(local-exchange)": 3922.0,
  "trade_latency_us_median": 4151.0,
  "latency_offset_trade_minus_book_us_median": 229.0,
  "book_latency_us_p99": 8111.0,
  "trade_latency_us_p99": 256007.93000000145
 },
 "trade_diagnostic": {
  "pruned1000": {
   "book_variant": "pruned1000",
   "trades_sampled": 300137,
   "sampling_stride": 8,
   "trades_without_prior_book_state": 5,
   "trad

## 10 · Processing requirements and the three test groups (kept apart on purpose)

* **E — notebook execution tests:** the code behaved (nothing pre-existing touched, only the intended files, deterministic replay, inspection only at completed messages, comparison function correct on constructed examples).
* **D — dataset integrity tests:** the downloaded files are intact and well-formed. **Any D failure prevents the folder from being finalised.**
* **R — reconstruction assessment:** computed in the next section against the frozen criteria. **Passing E and D says nothing about R.**

In [ ]:
REQ = {'download_total_mb': round(sum(d['bytes'] for d in DL.values()) / 1e6, 1), 'uncompressed_total_gb': round(sum(d['uncompressed_bytes'] for d in DL.values()) / 1e9, 2), 'rows_incremental_book_L2': SC['rows'], 'rows_trades': TR['rows'], 'peak_memory_gb': round(peak_mem_gb(), 2), 'stage_seconds': STAGES,
       'replay_seconds_both_variants': round(RP['seconds'], 1), 'replay_rows_per_second': round(RP['stats']['rows'] / max(RP['seconds'], 1)),
       'estimate_note': 'ETH and extra days will take roughly the same time/space per day ONLY IF their row counts are similar; this is an extrapolation, not a measurement.'}
print(json.dumps(REQ, indent=1, default=jsonable)); REPORT['requirements'] = REQ
TESTS = {'E': [], 'D': []}
def tcase(group, name, fn):
    try: ok, msg = fn()
    except Exception as e: ok, msg = False, f'EXC {type(e).__name__}: {str(e)[:200]}'
    TESTS[group].append((name, bool(ok), msg)); print(('PASS ' if ok else 'FAIL ') + name + ' | ' + str(msg))
# ---------------- E: notebook execution tests
def e01():
    now = {str(p.relative_to(BASE)): (p.stat().st_size, p.stat().st_mtime_ns) for p in BASE.rglob('*') if p.is_file() and not str(p.relative_to(BASE)).startswith(STAGE.name)}
    bad = [k for k in PIN0 if now.get(k) != PIN0[k]]; new = [k for k in now if k not in PIN0]; return (not bad and not new), (f'{len(PIN0)} pre-existing files unchanged, no new files outside staging' if not bad and not new else f'CHANGED {bad[:3]} NEW {new[:3]}')
def e02(): names = sorted(p.name for p in RAW.iterdir()); ok = all(SYMBOL in n and DAY in n and n.endswith('.csv.gz') for n in names) and len(names) == len(DL); return ok, f'{names}'
def e03():
    a = run_replay(max_rows=200_000, run_unpruned=True, samp=None, collect=False, verbose=False); b = run_replay(max_rows=200_000, run_unpruned=True, samp=None, collect=False, verbose=False); return a['fp'] == b['fp'] and a['stats']['messages'] == b['stats']['messages'], f"two runs identical, fingerprints {a['fp']}"
def e04(): return RP['stats']['checks'] == RP['stats']['messages'] == n_msg, f'book inspected exactly once per completed message ({n_msg:,}); never mid-message'
def e05():
    bk = Book(0); bids = [(100.0 - i * 0.1, 1.0 + i) for i in range(25)]; asks = [(100.1 + i * 0.1, 2.0 + i) for i in range(25)]
    for p, q in bids: bk.upd(True, p, q)
    for p, q in asks: bk.upd(False, p, q)
    snap = (np.array([p for p, _ in asks]), np.array([q for _, q in asks]), np.array([p for p, _ in bids]), np.array([q for _, q in bids])); r = compare_top(bk, snap); ok1 = all(r[N]['p'] == r[N]['lv'] == r[N]['q'] for N in TOPNS) and r['best_ok'] and r['spread_equal']
    bk.upd(True, bids[7][0], 99.0); r2 = compare_top(bk, snap); ok2 = r2[5]['all_pq'] and (not r2[10]['all_pq']) and r2[10]['p'] == r2[10]['lv'] and r2[10]['q'] == r2[10]['lv'] - 1
    snap2 = (snap[0], snap[1], snap[2].copy(), snap[3]); snap2[2][3] += 0.05; bk2 = Book(0); [bk2.upd(True, p, q) for p, q in bids]; [bk2.upd(False, p, q) for p, q in asks]; r3 = compare_top(bk2, snap2); ok3 = (not r3[5]['all_p']) and r3[5]['p'] == r3[5]['lv'] - 1
    return ok1 and ok2 and ok3, f'identical book -> 100%; one changed quantity at level 8 -> top-5 exact, top-10 quantity 9/10; one shifted price -> top-5 price 9/10 (checks {ok1},{ok2},{ok3})'
def e06():
    c = RP['cmp']['pruned1000']; matched = c['checkpoints'] + c['excluded_crossed_or_locked'] + c['empty_book_skipped']; return (not SAMP) or matched <= len(SAMP), f'matched {matched} <= distinct sampled keys {len(SAMP)}; unmatched rows are reported, never forced onto neighbouring messages'
for nm, fn in (('E01 pre-existing research files unchanged', e01), ('E02 only BTCUSDT 2024-05-01 files downloaded', e02), ('E03 replay deterministic (200k rows, 2 runs)', e03), ('E04 crossed-book check only at completed messages', e04), ('E05 snapshot-comparison function correct on constructed books', e05), ('E06 snapshot alignment accounting consistent', e06)): tcase('E', nm, fn)
# ---------------- D: dataset integrity tests
def d01(): bad = [k for k, d in DL.items() if not d['gzip_ok'] or (PROBE[k]['bytes'] and PROBE[k]['bytes'] != d['bytes'])]; return not bad, ('sizes equal server Content-Length and every gzip stream decodes fully' if not bad else bad)
def d02(): return all(v['matches_documented_header'] for v in SCHEMA.values()), 'headers equal the documented Tardis headers'
def d03(): bad = [(k, DL[k]['lines_incl_header'] - 1, n) for k, n in (('incremental_book_L2', SC['rows']), ('trades', TR['rows'])) if DL[k]['lines_incl_header'] - 1 != n]; return not bad, ('streamed row counts == independent newline counts' if not bad else bad)
def d04():
    df = pd.read_csv(DL['trades']['path'], nrows=300_000, usecols=['timestamp', 'local_timestamp', 'price', 'amount']); ok = df.price.min() > 0 and df.amount.min() > 0 and DAY0_US - 86_400_000_000 <= int(df.timestamp.min()) and int(df.timestamp.max()) <= DAY1_US + 86_400_000_000
    return ok, f'pandas recompute on the first 300k trades: price>0, amount>0, timestamps within +-1 day of {DAY}'
def d05():
    pre = pd.read_csv(DL['incremental_book_L2']['path'], nrows=500_000, usecols=['is_snapshot']).is_snapshot; first = int(np.argmax(pre.to_numpy())) if pre.any() else None; sb = SC['snapshot_batches']
    return (first is None and not sb) or bool(sb and (first is None or sb[0]['start_row'] == first)), f'first snapshot row: pandas {first} vs scan {sb[0]["start_row"] if sb else None}'
def d06(): return SC['symbols'] == {SYMBOL: SC['rows']} and set(SC['exchanges']) == {EXCHANGE}, f"book symbols {SC['symbols']} exchanges {SC['exchanges']}"
def d07():
    tot = SC['rows'] + TR['rows']; out = SC['local_outside_day_rows'] + TR['local_outside_day_rows']; return SC['timestamp_unit_check_ok'] and out / max(tot, 1) <= 0.001, f'timestamps are microseconds; local_timestamp outside {DAY} UTC: {out:,} of {tot:,} rows ({out/max(tot,1):.4%}, limit 0.1%)'
def d08(): bad = {'book_nan_cells': SC['nan_cells'], 'book_negative_amount': SC['amount_negative'], 'book_price<=0': SC['price_le_zero'], 'trade_amount_min': TR['amount_min'], 'trade_price_min': TR['price_range'][0]}; ok = SC['nan_cells'] == 0 and SC['amount_negative'] == 0 and SC['price_le_zero'] == 0 and TR['amount_min'] > 0 and TR['price_range'][0] > 0; return ok, str(bad)
def d09(): ok = set(SC['sides']) <= {'bid', 'ask'} and set(TR['sides']) <= {'buy', 'sell', 'unknown'}; return ok, f"book sides {SC['sides']} | trade sides {TR['sides']}"
def d10(): sb = SC['snapshot_batches']; ok = bool(sb) and all(b['bid_rows'] > 0 and b['ask_rows'] > 0 for b in sb); return ok, f'{len(sb)} snapshot batch(es); every batch has both bid and ask rows' if ok else f'snapshot batches missing or one-sided: {sb[:3]}'
for nm, fn in (('D01 file sizes + gzip integrity', d01), ('D02 schema equals documentation', d02), ('D03 row counts: streamed vs newline count', d03), ('D04 independent pandas recompute on 300k trades', d04), ('D05 first snapshot row vs pandas', d05), ('D06 symbol/exchange columns constant', d06), ('D07 microsecond units and day coverage', d07), ('D08 no NaN/negative/non-positive values', d08), ('D09 side vocabulary', d09), ('D10 snapshot batches contain both sides', d10)): tcase('D', nm, fn)
OBS = {'book_exchange_ts_inversions': SC['ts_inversions'], 'book_local_ts_inversions': SC['local_inversions'], 'trade_local_ts_inversions': TR['local_inversions'], 'book_zero_amount_rows(level_removals)': SC['amount_zero_rows'], 'trade_id_gaps(>1)': TR['id_gaps(>1)'], 'trade_id_dup_or_backwards': TR['id_dup_or_backwards'],
       'note': 'Observations, not pass/fail. The replay uses file order, as Tardis documents (rows are in capture order); inversions are listed so you can judge.'}
print('\nDataset observations (not pass/fail):', OBS); REPORT['observations'] = OBS
print(f"\nE (execution): {sum(t[1] for t in TESTS['E'])}/{len(TESTS['E'])} | D (dataset integrity): {sum(t[1] for t in TESTS['D'])}/{len(TESTS['D'])} | R (reconstruction): see next section")

{
 "download_total_mb": 271.5,
 "uncompressed_total_gb": 3.15,
 "rows_incremental_book_L2": 31005311,
 "rows_trades": 2401094,
 "peak_memory_gb": 2.52,
 "stage_seconds": {
  "fingerprint": {
   "seconds": 0.9,
   "peak_mem_gb_so_far": 0.18
  },
  "download": {
   "seconds": 4.7,
   "peak_mem_gb_so_far": 0.21
  },
  "integrity": {
   "seconds": 17.5,
   "peak_mem_gb_so_far": 0.25
  },
  "scan_incremental": {
   "seconds": 33.3,
   "peak_mem_gb_so_far": 2.08
  },
  "scan_trades": {
   "seconds": 2.4,
   "peak_mem_gb_so_far": 2.08
  },
  "load_snapshot25_sample": {
   "seconds": 16.9,
   "peak_mem_gb_so_far": 2.08
  },
  "replay": {
   "seconds": 443.5,
   "peak_mem_gb_so_far": 2.52
  },
  "trade_diagnostic": {
   "seconds": 16.9,
   "peak_mem_gb_so_far": 2.52
  }
 },
 "replay_seconds_both_variants": 442.6,
 "replay_rows_per_second": 70046,
 "estimate_note": "ETH and extra days will take roughly the same time/space per day ONLY IF their row counts are similar; this is an extrapolation, no

## 11 · Reconstruction assessment (group R) against the criteria frozen before any data was seen

These criteria judge **only** the order-book reconstruction. They never decide whether the folder is saved (that depends on groups E and D alone), and the trade-price diagnostic of section 9 is **not** one of them. A "CONSISTENT" result means agreement with the provider's *own* `book_snapshot_25`, from the same feed — it is **not** independent validation and **not** a scientific validation of the book.

In [ ]:
R = {}
HAVE_S25 = 'book_snapshot_25' in DL
for name in CROSS:
    c = CROSS[name]; m = CMP.get(name, {}); p25 = m.get('top25_price_agreement_(levels)'); q25 = m.get('top25_quantity_agreement_(price AND qty equal)'); nck = m.get('ALIGNED_CHECKPOINTS', 0)
    R[name] = {
     'R01 crossed+locked share <= 0.5%': (c['crossed_plus_locked_share'] <= 0.005, c['crossed_plus_locked_share']),
     'R02 top-25 price agreement >= 99% (aligned checkpoints)': ((p25 or 0) >= 0.99 if HAVE_S25 else None, p25),
     'R03 top-25 quantity agreement >= 99% (aligned checkpoints)': ((q25 or 0) >= 0.99 if HAVE_S25 else None, q25),
     'R04 aligned snapshot checkpoints >= 500': (nck >= 500 if HAVE_S25 else None, nck),
     'R05 >= 99% of the 1,440 minutes contain a book message': (cov['coverage_of_day_by_minutes'] >= 0.99, cov['coverage_of_day_by_minutes'])}
def r_verdict(name):
    d = R[name]; fails = [k for k, (ok, v) in d.items() if ok is False]; untested = [k for k, (ok, v) in d.items() if ok is None]
    if untested: return 'UNTESTED for ' + '; '.join(untested) + ' (book_snapshot_25 not downloaded)' + (' | FAILED: ' + '; '.join(fails) if fails else '')
    return ("CONSISTENT with the provider's own book_snapshot_25 (same provider/feed; NOT independent validation)" if not fails else 'NOT CONSISTENT -> failed: ' + '; '.join(fails))
RV = {k: r_verdict(k) for k in R}
REPORT['reconstruction_criteria'] = {k: {c: {'pass': ok, 'value': v} for c, (ok, v) in d.items()} for k, d in R.items()}; REPORT['reconstruction_verdict'] = RV
for k in R:
    print(k); [print('  ' + ('PASS ' if ok else ('FAIL ' if ok is False else 'N/A  ')) + c + ' -> ' + str(v)) for c, (ok, v) in R[k].items()]; print('  VERDICT:', RV[k])

pruned1000
  PASS R01 crossed+locked share <= 0.5% -> 0.0
  FAIL R02 top-25 price agreement >= 99% (aligned checkpoints) -> 0.9838254602892541
  FAIL R03 top-25 quantity agreement >= 99% (aligned checkpoints) -> 0.9838254602892541
  PASS R04 aligned snapshot checkpoints >= 500 -> 20259
  PASS R05 >= 99% of the 1,440 minutes contain a book message -> 1.0
  VERDICT: NOT CONSISTENT -> failed: R02 top-25 price agreement >= 99% (aligned checkpoints); R03 top-25 quantity agreement >= 99% (aligned checkpoints)
unpruned
  PASS R01 crossed+locked share <= 0.5% -> 0.0
  PASS R02 top-25 price agreement >= 99% (aligned checkpoints) -> 1.0
  PASS R03 top-25 quantity agreement >= 99% (aligned checkpoints) -> 1.0
  PASS R04 aligned snapshot checkpoints >= 500 -> 20259
  PASS R05 >= 99% of the 1,440 minutes contain a book message -> 1.0
  VERDICT: CONSISTENT with the provider's own book_snapshot_25 (same provider/feed; NOT independent validation)


## 12 · FINAL DATASET AUDIT SHARE BLOCK (copy everything between the lines and paste it back to Claude)

In [ ]:
ghost = {k: {'levels_end': REPORT['depth_series_summary'].get(k, {}).get('levels_end'), 'levels_max': REPORT['depth_series_summary'].get(k, {}).get('levels_max'), 'far_levels_max': REPORT['depth_series_summary'].get(k, {}).get('far_levels_max_(>5pct_from_mid)')} for k in DEPTH}
unc = ['One symbol and one day only: this says nothing about other days, ETHUSDT, or whether the free-day files are representative.',
       'The CSV has NO update-ID columns, so exchange sequence continuity (U/u) cannot be re-audited here; Tardis states it validates U/u and restarts its connection on a gap (documentation claim, not verified by this notebook).',
       "book_snapshot_25 is produced by Tardis from the same feed, so agreement with it shows consistency with Tardis' own reconstruction, not independence from the exchange or scientific validation.",
       'The trade and depth streams are separate WebSocket feeds; the trade-vs-book figures are a diagnostic of plausibility and timing, not proof of exact ordering, and they gate nothing.',
       'The terms do not say whether publishing thesis results from this data is allowed; ask Tardis in writing. Raw files and trained models must not be given to third parties.']
L = []; P = L.append
def fm(x, f='{:.4%}'): return f.format(x) if isinstance(x, (int, float)) and x == x else str(x)
P('=' * 100); P('FINAL DATASET AUDIT SHARE BLOCK - Tardis.dev Binance Spot BTCUSDT 2024-05-01 (v1)'); P('=' * 100)
P(f'run: {RUN_STARTED} | folder: {OUT_FINAL} | free-access download, no API key: {API_KEY is None}')
P('Three kinds of result are kept apart below: [E] notebook execution tests, [D] dataset integrity tests, [R] order-book reconstruction assessment.')
P('PASSING E AND D DOES NOT MEAN THE RECONSTRUCTED BOOK IS VALIDATED. Only section [R] speaks about the book, and only as consistency with the provider\'s own snapshots.')
P('\n[A] FILES'); [P(f"  {d['file']}: {d['bytes']/1e6:,.1f} MB compressed | {d['uncompressed_bytes']/1e9:.2f} GB uncompressed | lines {d['lines_incl_header']-1:,} | gzip_ok {d['gzip_ok']} | sha256 {d['sha256'][:16]}...") for d in DL.values()]
P('  schemas match documentation: ' + str({k: v['matches_documented_header'] for k, v in SCHEMA.items()}))
P('\n[B] TIMESTAMPS (microseconds)'); P(f"  book exchange ts: {SC['timestamp_min']} -> {SC['timestamp_max']} | local ts: {SC['local_min']} -> {SC['local_max']} | inversions ts/local: {SC['ts_inversions']}/{SC['local_inversions']}")
P(f"  trades local ts: {TR['local_min']} -> {TR['local_max']} | trade local-ts inversions {TR['local_inversions']} | trade id gaps {TR['id_gaps(>1)']} dup/back {TR['id_dup_or_backwards']} | latency (local-exchange) trades median {TR['local_minus_exchange_us_median']:.0f} us p99 {TR['local_minus_exchange_us_p99']:.0f} us | book median {SC['book_local_minus_exchange_us_median']:.0f} us p99 {SC['book_local_minus_exchange_us_p99']:.0f} us")
P(f"  book rows {SC['rows']:,} | trades {TR['rows']:,} | side values {SC['sides']} | zero-amount rows {SC['amount_zero_rows']:,} | negative {SC['amount_negative']} | NaN cells {SC['nan_cells']} | price<=0 {SC['price_le_zero']}")
P('\n[C] COVERAGE'); P('  ' + json.dumps(cov, default=jsonable)); P(f"  book message gaps (local ts, row-level) >1s/10s/60s/300s: {SC['gap_counts']} | largest: {SC['largest_gaps_s'][:5]}")
P('\n[D-snap] SNAPSHOT BOUNDARIES'); P(f"  rows before first snapshot (to be skipped): {SC['rows_before_first_snapshot']:,} | snapshot rows total {SC['snapshot_rows']:,} | snapshot batches: {len(SC['snapshot_batches'])}")
for b in SC['snapshot_batches'][:12]: P(f"    start_row {b['start_row']:,} at {iso_us(b['local_ts'])}: {b['rows']:,} rows (bids {b['bid_rows']:,} / asks {b['ask_rows']:,})")
P('\n' + '-' * 100); P('[E] NOTEBOOK EXECUTION TESTS (did the code behave?)'); [P(f"  {'PASS' if ok else 'FAIL'} {n} | {m}") for n, ok, m in TESTS['E']]
P(f"  => E: {sum(t[1] for t in TESTS['E'])}/{len(TESTS['E'])} passed")
P('\n' + '-' * 100); P('[D] DATASET INTEGRITY TESTS (are the files intact and well-formed?)'); [P(f"  {'PASS' if ok else 'FAIL'} {n} | {m}") for n, ok, m in TESTS['D']]
P(f"  => D: {sum(t[1] for t in TESTS['D'])}/{len(TESTS['D'])} passed"); P('  observations (not pass/fail): ' + json.dumps(OBS, default=jsonable))
P('\n' + '-' * 100); P('[R] ORDER-BOOK RECONSTRUCTION ASSESSMENT (independent of E and D; same-provider consistency only)')
for k in CROSS: P(f"  {k}: messages {CROSS[k]['messages_checked']:,} | crossed {CROSS[k]['crossed_messages']:,} ({CROSS[k]['crossed_share']:.4%}) | locked {CROSS[k]['locked_messages']:,} ({CROSS[k]['locked_share']:.4%}) | crossed+locked {CROSS[k]['crossed_plus_locked_share']:.4%} | longest crossed run {CROSS[k]['max_consecutive_crossed']} | final levels {RP['final_levels'][k]} | resets {RP['stats']['resets']}")
P('  snapshot comparison rule: book_snapshot_25 row vs rebuilt book after the completed message with EXACTLY equal local_timestamp; unmatched rows counted, crossed/locked/empty moments excluded and counted.')
for k, m in CMP.items():
    P(f"  vs book_snapshot_25 [{k}]: rows {m['snapshot25_rows_total']:,} (sampled {m['snapshot25_rows_sampled']}) | matched to a completed message {m['matched_to_a_completed_message']:,} | unmatched {m['unmatched_no_message_with_equal_local_timestamp']:,} | excluded crossed/locked {m['excluded_book_crossed_or_locked']:,} | excluded empty {m['excluded_book_empty']:,} | ALIGNED CHECKPOINTS {m['ALIGNED_CHECKPOINTS']:,}")
    for N in TOPNS: P(f"     top-{N}: price agreement {fm(m[f'top{N}_price_agreement_(levels)'])} | quantity agreement (price+qty) {fm(m[f'top{N}_quantity_agreement_(price AND qty equal)'])} | qty given price match {fm(m[f'top{N}_quantity_given_price_match'])} | checkpoints all-prices-exact {fm(m[f'top{N}_checkpoints_all_prices_exact'])} / all-prices+qty-exact {fm(m[f'top{N}_checkpoints_all_prices_and_qty_exact'])}")
    P(f"     best bid&ask equal {fm(m['best_bid_and_ask_both_equal'])} | spread exactly equal {fm(m['spread_exactly_equal'])} | |spread diff| median {fm(m['spread_abs_diff_bps_median'], '{:.4f}')} bps p99 {fm(m['spread_abs_diff_bps_p99'], '{:.4f}')} bps | worst hours {m['worst_hours_(share_of_checkpoints_all25_exact)']}")
P('  stale-level indicators (levels per side end / max / max beyond 5% of mid): ' + json.dumps(ghost, default=jsonable)); P('  depth/spread summary: ' + json.dumps(REPORT['depth_series_summary'], default=jsonable))
P('  frozen reconstruction criteria:')
for k, d in R.items():
    P(f'   variant {k}:'); [P(f"     {'PASS' if ok else ('FAIL' if ok is False else 'N/A ')}  {c}  -> {v}") for c, (ok, v) in d.items()]; P('     RECONSTRUCTION VERDICT: ' + RV[k])
P('\n' + '-' * 100); P('[T] TRADES vs BOOK - DIAGNOSTIC ONLY (not a criterion; never fails anything)'); P('  timestamp compatibility: ' + json.dumps(TS_COMPAT, default=jsonable))
for k, t in TD.items(): P(f'  {k}: ' + json.dumps(t, default=jsonable))
P('\n[G] PROCESSING REQUIREMENTS'); P('  ' + json.dumps(REQ, default=jsonable))
P('\n[I] LICENCE REMINDER: internal research/educational use; no redistribution of raw data; no giving trained models to third parties; publication of results not addressed in the terms -> ask Tardis in writing.')
P('\nUNCERTAINTIES'); [P('  - ' + u) for u in unc]
GATE_OK_E = all(t[1] for t in TESTS['E']); GATE_OK_D = all(t[1] for t in TESTS['D'])
P(f"\nFOLDER GATE (E and D only): E {'ok' if GATE_OK_E else 'FAILED'} | D {'ok' if GATE_OK_D else 'FAILED'} -> the R verdict does not decide whether files are saved.")
P('Nothing else was downloaded (no other dates, no ETHUSDT). No synthetic events, graphs or models were created. Existing research artifacts were not modified. WAITING FOR YOUR APPROVAL before any next step.'); P('=' * 100)
SHARE = '\n'.join(L)
STAGE.mkdir(exist_ok=True); (STAGE / 'reports').mkdir(exist_ok=True)
REPORT['tests'] = TESTS; REPORT['unc'] = unc; REPORT['finished'] = UTC()
dump_json(REPORT, STAGE / 'reports' / 'audit_report.json'); (STAGE / 'reports' / 'share_block.txt').write_text(SHARE, encoding='utf-8')
pd.DataFrame(SC['snapshot_batches']).to_csv(STAGE / 'reports' / 'snapshot_batches.csv', index=False)
for k, df in DEPTH.items(): df.to_parquet(STAGE / 'reports' / f'depth_series_{k}.parquet', index=False)
MAN = {str(p.relative_to(STAGE)): sha256_file(p) for p in sorted(STAGE.rglob('*')) if p.is_file()}; dump_json(MAN, STAGE / 'manifest.json')
okm = all(sha256_file(STAGE / k) == v for k, v in MAN.items()); all_ok = GATE_OK_E and GATE_OK_D and okm
if all_ok: os.rename(STAGE, OUT_FINAL); print('\nSAVED ->', OUT_FINAL, '(files intact; this is NOT a statement that the book is validated)')
else: print('\n!! E or D tests FAILED (or manifest mismatch) -> staging kept at', STAGE, '(not renamed). Share this block anyway.')
print(SHARE); print(f'manifest verified: {okm} ({len(MAN)} files)')


SAVED -> /content/drive/MyDrive/research datasets/TGNN_Tardis_Binance_Spot_BTCUSDT_2024-05-01_v1 (files intact; this is NOT a statement that the book is validated)
FINAL DATASET AUDIT SHARE BLOCK - Tardis.dev Binance Spot BTCUSDT 2024-05-01 (v1)
run: 2026-10-12T06:12:08.512646+00:00 | folder: /content/drive/MyDrive/research datasets/TGNN_Tardis_Binance_Spot_BTCUSDT_2024-05-01_v1 | free-access download, no API key: True
Three kinds of result are kept apart below: [E] notebook execution tests, [D] dataset integrity tests, [R] order-book reconstruction assessment.
PASSING E AND D DOES NOT MEAN THE RECONSTRUCTED BOOK IS VALIDATED. Only section [R] speaks about the book, and only as consistency with the provider's own snapshots.

[A] FILES
  binance_incremental_book_L2_2024-05-01_BTCUSDT.csv.gz: 183.1 MB compressed | 2.28 GB uncompressed | lines 31,005,311 | gzip_ok True | sha256 5173777ef4dc7418...
  binance_trades_2024-05-01_BTCUSDT.csv.gz: 27.9 MB compressed | 0.20 GB uncompressed | lin